In [1]:
#!/usr/bin/env python3
"""
Test script for the integration examples to verify they work correctly.
"""

import sys
import os
import traceback
import importlib

from llm_expert_base import (
    create_reasoning_llm_expert,
    create_creative_llm_expert,
    create_systematic_llm_expert
)
from llm_expert_and_task import EnhancedTask
from marco_enhanced_core import CognitiveStateSpace
import json

def test_marco_integration():
    """Test the marco_integration_example module"""
    print("🧪 Testing marco_integration_example...")
    
    try:
        from marco_integration_example import MARCOSystem, create_sample_task, run_marco_demo
        
        # Test basic system creation
        config = {
            "default_compute_budget": 50.0,
            "default_time_budget": 15.0,
            "default_memory_budget": 4.0,
            "target_confidence": 0.8,
            "max_iterations": 3,
            "training_mode": True,
            "use_mock_llm": True
        }
        
        marco = MARCOSystem(config)
        print(f"  ✅ MARCOSystem created with {len(marco.expert_modules)} experts")
        
        # Test task creation
        task = create_sample_task()
        print(f"  ✅ Sample task created: {task['task_id']}")
        
        # Test task solving
        from marco_enhanced_core import ResourceBudget
        budget = ResourceBudget(
            max_compute=30.0,
            max_time=10.0,
            max_memory=4.0,
            target_confidence=0.7
        )
        
        result = marco.solve_task(task, budget)
        print(f"  ✅ Task solved with accuracy: {result['results']['accuracy']:.3f}")
        
        # Test performance summary
        summary = marco.get_performance_summary()
        print(f"  ✅ Performance summary generated: {summary['system_performance']['total_tasks_solved']} tasks")
        
        return True
        
    except Exception as e:
        print(f"  ❌ Error in marco_integration_example: {e}")
        traceback.print_exc()
        return False

def test_llm_expert_base():
    """Test the core LLM expert functionality"""
    print("\n🧪 Testing llm_expert_base...")
    
    try:
        # Create experts
        experts = {
            "reasoning": create_reasoning_llm_expert(),
            "creative": create_creative_llm_expert(),
            "systematic": create_systematic_llm_expert()
        }
        print(f"  ✅ Created {len(experts)} LLM experts")
        
        # Create test task
        task = EnhancedTask(
            "test_task",
            [([[1, 0], [0, 1]], [[1, 0], [0, 1]])],
            [([[2, 0], [0, 2]], [[2, 0], [0, 2]])]
        )
        css = CognitiveStateSpace()
        
        # Test each expert
        all_hypotheses = []
        for name, expert in experts.items():
            
            hypotheses = expert.generate_hypotheses(task, css)
            print(f"  ✅ {name} expert generated {len(hypotheses)} hypotheses")
            
            for hyp in hypotheses:
                css.add_hypothesis(hyp)
                all_hypotheses.append(hyp)
                print(hyp.content)

        # Test evaluation
        for name, expert in experts.items():
            beliefs = expert.evaluate_hypotheses(task, css, all_hypotheses)
            print(f"  ✅ {name} expert evaluated {len(beliefs)} hypotheses")
            print(beliefs)
        
        return True
        
    except Exception as e:
        print(f"  ❌ Error in llm_expert_base: {e}")
        traceback.print_exc()
        return False

def test_enhanced_mcu():
    """Test the enhanced MCU functionality"""
    print("\n🧪 Testing enhanced_mcu...")
    
    try:
        from enhanced_mcu import EnhancedMetaControlUnit
        from llm_expert_base import create_reasoning_llm_expert
        
        # Create expert modules
        expert_modules = {
            "reasoning_llm": create_reasoning_llm_expert()
        }
        
        # Create MCU
        config = {
            "max_iterations": 3,
            "task_embedding_dim": 6,
            "default_compute_budget": 50.0
        }
        
        mcu = EnhancedMetaControlUnit(expert_modules, config)
        print(f"  ✅ EnhancedMetaControlUnit created with {len(expert_modules)} experts")
        
        # Test task feature extraction
        from llm_expert_and_task import EnhancedTask
        task = EnhancedTask(
            "test_task",
            [([[1, 0], [0, 1]], [[1, 0], [0, 1]])],
            [([[2, 0], [0, 2]], [[2, 0], [0, 2]])]
        )
        
        features = mcu.extract_task_features(task)
        print(f"  ✅ Task features extracted: shape {features.shape}")
        
        # Test resource prediction
        if hasattr(mcu, 'predict_resource_requirements'):
            complexity, compute, time_req, accuracy = mcu.predict_resource_requirements(task)
            print(f"  ✅ Resource prediction: complexity={complexity:.3f}, compute={compute:.1f}")
        
        return True
        
    except Exception as e:
        print(f"  ❌ Error in enhanced_mcu: {e}")
        traceback.print_exc()
        return False

def main():
    """Run all integration tests"""
    print("🚀 Starting Integration Tests...")
    print("=" * 50)
    
    tests = [
        ("LLM Expert Base", test_llm_expert_base),
        ("Enhanced MCU", test_enhanced_mcu),
        ("MARCO Integration", test_marco_integration),
    ]
    
    passed = 0
    total = len(tests)
    
    for test_name, test_func in tests:
        try:
            if test_func():
                print(f"✅ {test_name} - PASSED")
                passed += 1
            else:
                print(f"❌ {test_name} - FAILED")
        except Exception as e:
            print(f"❌ {test_name} - ERROR: {e}")
    
    print("\n" + "=" * 50)
    print(f"FINAL RESULTS: {passed}/{total} tests passed")
    
    if passed == total:
        print("🎉 All integration tests passed!")
        return 0
    else:
        print("⚠️ Some tests failed. Check the output above.")
        return 1

if __name__ == "__main__":
    exit_code = main()


ModuleNotFoundError: No module named 'llm_expert_base'